<a href="https://colab.research.google.com/github/smsag99/Thesis/blob/main/codes/HTS_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hierarchical Time Series Forecasting — Dairy Buffalo Milk Yield

**Hierarchy**: Total → Farm → Animal  
**Target**: `milk_kg` (daily milk yield per test-day record, resampled monthly)  
**Pipeline**: Aggregate → S-matrix → Base Forecasts → Reconciliation → Evaluation

## 0. Configuration

In [8]:
# ── Paths ────────────────────────────────────────────────────────────────────
# path = '/content/drive/MyDrive/Thesis_Data/'   # Colab
path = '../Thesis_Data/'                          # Local

DATA_FILE = path + 'Final_Data/Final_Merged_Data.csv'
OUT_DIR   = path + 'HTS_Results/'

# ── Modelling choices ────────────────────────────────────────────────────────
TARGET    = 'milk_kg'
FREQ      = 'MS'        # Month-Start — test-day records are ~monthly
H         = 3           # forecast horizon (months ahead)
N_WINDOWS = 3           # rolling cross-validation windows

# ── Hierarchy bottom level ────────────────────────────────────────────────────
# 'farm'   → Total > Farm (~300 series)           ← default, memory-safe
# 'animal' → Total > Farm > Animal (~80k series)  ← needs large RAM (>32 GB)
BOTTOM_LEVEL = 'farm'

# ── Reproducibility & memory ─────────────────────────────────────────────────
SEED           = 42

## 1. Install & Import

In [ ]:
%capture
!pip install hierarchicalforecast statsforecast

In [9]:
import os
import gc
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import scipy.sparse as sp

from statsforecast import StatsForecast
from statsforecast.models import AutoETS, AutoARIMA, Naive, SeasonalNaive

from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace
from hierarchicalforecast.evaluation import HierarchicalEvaluation

warnings.filterwarnings('ignore')
os.makedirs(OUT_DIR, exist_ok=True)

print('Libraries loaded ✅')

Libraries loaded ✅


## 2. Load Preprocessed Data

In [10]:
df = pd.read_csv(
    DATA_FILE,
    usecols=['Farm_Code', 'Animal_ID', 'dtt', TARGET],
    parse_dates=['dtt'],
)

df['Farm_Code']  = df['Farm_Code'].astype(str)
df['Animal_ID']  = df['Animal_ID'].astype(str)


print(f'Records  : {len(df):,}')
print(f'Animals  : {df["Animal_ID"].nunique():,}')
print(f'Farms    : {df["Farm_Code"].nunique():,}')
print(f'Date range: {df["dtt"].min().date()} → {df["dtt"].max().date()}')
df.head()

Records  : 671,655
Animals  : 81,119
Farms    : 314
Date range: 2013-01-02 → 2023-12-29


,Farm_Code,Animal_ID,dtt,milk_kg
0,521513,IT003990057639,2013-06-04,6.8
1,521513,IT003990057639,2013-07-02,8.0
2,521513,IT003990057639,2013-09-03,7.8
3,521513,IT003990057639,2013-10-02,8.2
4,521513,IT003990057639,2013-11-04,5.2


## 3. Resample to Monthly Frequency

Test-day records are irregular (~one per month per animal). We snap each record to the first day of its month (`MS`) and take the **mean** when multiple tests fall in the same month (rare after preprocessing).

In [13]:
# Snap test-day date to month start
df['ds'] = df['dtt'].dt.to_period('M').dt.to_timestamp()

# Aggregate: mean milk per animal per month
df_monthly = (
    df.groupby(['Farm_Code', 'Animal_ID', 'ds'])[TARGET]
    .mean()
    .reset_index()
    .rename(columns={TARGET: 'y'})
)

print(f'Monthly records (animal level): {len(df_monthly):,}')
print(f'Date range: {df_monthly["ds"].min().date()} → {df_monthly["ds"].max().date()}')
df_monthly.head()

Monthly records (animal level): 660,241
Date range: 2013-01-01 → 2023-12-01


,Farm_Code,Animal_ID,ds,y
0,1114232,IT019990445374,2013-01-01,8.0
1,1114232,IT019990445374,2013-02-01,7.4
2,1114232,IT019990445374,2013-03-01,6.0
3,1114232,IT019990445374,2013-04-01,7.4
4,1114232,IT019990445420,2013-01-01,11.6


## 4. Build the Hierarchical Panel (`Y_df`)

Three levels:

| Level | `unique_id` format | Example |
|---|---|---|
| Total | `Total` | `Total` |
| Farm  | `Farm/{Farm_Code}` | `Farm/521513` |
| Animal | `{Farm_Code}/{Animal_ID}` | `521513/IT003990094623` |

In [ ]:
# ── Bottom level: animal series ───────────────────────────────────────────────
animal_df = df_monthly.copy()
animal_df['unique_id'] = animal_df['Farm_Code'] + '/' + animal_df['Animal_ID']
animal_df = animal_df[['unique_id', 'ds', 'y']]

# ── Middle level: farm aggregates ─────────────────────────────────────────────
farm_df = (
    df_monthly.groupby(['Farm_Code', 'ds'])['y']
    .sum()
    .reset_index()
)
farm_df['unique_id'] = 'Farm/' + farm_df['Farm_Code']
farm_df = farm_df[['unique_id', 'ds', 'y']]

# ── Top level: grand total ─────────────────────────────────────────────────────
total_df = (
    df_monthly.groupby('ds')['y']
    .sum()
    .reset_index()
)
total_df['unique_id'] = 'Total'
total_df = total_df[['unique_id', 'ds', 'y']]

# ── Combine ────────────────────────────────────────────────────────────────────
Y_df = pd.concat([total_df, farm_df, animal_df], ignore_index=True)
Y_df = Y_df.sort_values(['unique_id', 'ds']).reset_index(drop=True)

n_total  = 1
n_farms  = farm_df['unique_id'].nunique()
n_animals = animal_df['unique_id'].nunique()

print(f'Total series : {Y_df["unique_id"].nunique():,}')
print(f'  Top (Total): {n_total}')
print(f'  Farm level : {n_farms}')
print(f'  Animal lvl : {n_animals:,}')
print(f'Total rows in Y_df: {len(Y_df):,}')
Y_df.head(5)

## 5. Build the Hierarchical Panel (`Y_df`), Summing Matrix (`S_df`), and `tags`

We use `hierarchicalforecast.utils.aggregate` — the standard way to build all three outputs at once.  
It handles the sparse S matrix internally so it won't blow up memory.

**Hierarchy options (set `BOTTOM_LEVEL` in config):**

| `BOTTOM_LEVEL` | Levels | Series count | RAM |
|---|---|---|---|
| `'farm'` | Total → Farm | ~300 | < 1 GB ✅ |
| `'animal'` | Total → Farm → Animal | ~80k | > 32 GB ⚠️ |

In [19]:
from hierarchicalforecast.utils import aggregate

# Prepare a flat DataFrame with one column per hierarchy level + ds + y
df_hier = df_monthly.head(100).copy()
df_hier['Total'] = 'Total'
df_hier.head()
spec = [
    ['Total'],
    ['Total', 'Farm_Code'],
    ['Total', 'Farm_Code', 'Animal_ID'],
]
# aggregate() returns Y_df (long panel), S_df (summing matrix), tags (level → ids)
Y_df, S_df, tags = aggregate(df_hier, spec)


In [27]:
n_series = Y_df['unique_id'].nunique()
print(f'BOTTOM_LEVEL : {BOTTOM_LEVEL}')
print(f'Total series : {n_series:,}')
for level, ids in tags.items():
    print(f'  {level:<12}: {len(ids):,} series')
print(f'S_df shape   : {S_df.shape}')
print(f'Y_df rows    : {len(Y_df):,}')
print(f'Date range   : {Y_df["ds"].min().date()} → {Y_df["ds"].max().date()}')

BOTTOM_LEVEL : farm
Total series : 20
  Total       : 1 series
  Total/Farm_Code: 1 series
  Total/Farm_Code/Animal_ID: 18 series
S_df shape   : (20, 19)
Y_df rows    : 128
Date range   : 2013-01-01 → 2014-03-01


In [ ]:

# Convenience lists used in later cells
all_ids    = Y_df['unique_id'].unique().tolist()
bottom_ids = tags[list(tags.keys())[-1]]   # last level = bottom
farm_ids   = tags[list(tags.keys())[-2]]   # last level = bottom

In [ ]:
from hierarchicalforecast.utils import aggregate

# Prepare a flat DataFrame with one column per hierarchy level + ds + y
df_hier = df_monthly.copy()
df_hier['Total'] = 'Total'

if BOTTOM_LEVEL == 'farm':
    # 2-level: Total → Farm
    spec = [
        ['Total'],
        ['Total', 'Farm_Code'],
    ]
    df_agg = df_hier.groupby(['Total', 'Farm_Code', 'ds'])['y'].sum().reset_index()

elif BOTTOM_LEVEL == 'animal':
    # 3-level: Total → Farm → Animal
    # Make Animal unique across farms by prefixing Farm_Code
    df_hier['Animal'] = df_hier['Farm_Code'] + '/' + df_hier['Animal_ID']
    spec = [
        ['Total'],
        ['Total', 'Farm_Code'],
        ['Total', 'Farm_Code', 'Animal'],
    ]
    df_agg = df_hier[['Total', 'Farm_Code', 'Animal', 'ds', 'y']].copy()

else:
    raise ValueError(f"BOTTOM_LEVEL must be 'farm' or 'animal', got: {BOTTOM_LEVEL}")

# aggregate() returns Y_df (long panel), S_df (summing matrix), tags (level → ids)
Y_df, S_df, tags = aggregate(df_agg, spec)

# Rename 'y' column to match statsforecast expectation
# (aggregate already names it 'y', just confirming)
Y_df = Y_df.reset_index(drop=True)

# Print summary
n_series = Y_df['unique_id'].nunique()
print(f'BOTTOM_LEVEL : {BOTTOM_LEVEL}')
print(f'Total series : {n_series:,}')
for level, ids in tags.items():
    print(f'  {level:<12}: {len(ids):,} series')
print(f'S_df shape   : {S_df.shape}')
print(f'Y_df rows    : {len(Y_df):,}')
print(f'Date range   : {Y_df["ds"].min().date()} → {Y_df["ds"].max().date()}')

# Convenience lists used in later cells
all_ids    = Y_df['unique_id'].unique().tolist()
bottom_ids = tags[list(tags.keys())[-1]]   # last level = bottom
farm_ids   = tags['Farm_Code'] if 'Farm_Code' in tags else []

## 6. Verify Coherence

At any given date, `Total` must equal the sum of all `Farm` series, which must equal the sum of all `Animal` series.

In [ ]:
# Pick a date in the middle of the series to verify
check_date = sorted(Y_df['ds'].unique())[len(Y_df['ds'].unique()) // 2]
snap = Y_df[Y_df['ds'] == check_date].set_index('unique_id')['y']

total_val  = snap.reindex(['Total']).sum()
bottom_sum = snap.reindex(bottom_ids).sum()

print(f'Coherence check at {check_date.date()}:')
print(f'  Total series value    : {total_val:,.2f}')
print(f'  Sum of bottom series  : {bottom_sum:,.2f}')
print(f'  Coherent              : {np.isclose(total_val, bottom_sum)}')

## 7. Visualise the Hierarchy

Plot the Total series and a sample of farm-level series to inspect seasonality and trend.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Top: Total
tot = Y_df[Y_df['unique_id'] == 'Total'].sort_values('ds')
axes[0].plot(tot['ds'], tot['y'], color='#1a1a2e', linewidth=2)
axes[0].set_title('Total Herd — Monthly Milk Yield', fontsize=13, fontweight='bold')
axes[0].set_ylabel('kg')

# Middle: random sample of 5 farms
np.random.seed(SEED)
sample_farms_plot = np.random.choice(farm_ids, min(5, len(farm_ids)), replace=False)
for fid in sample_farms_plot:
    ts = Y_df[Y_df['unique_id'] == fid].sort_values('ds')
    axes[1].plot(ts['ds'], ts['y'], label=fid.replace('Farm/', ''), linewidth=1)
axes[1].set_title('Sample of Farm-Level Series (5 farms)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('kg')
axes[1].legend(fontsize=7, ncol=3)

# Bottom: random sample of 5 animals from first sample farm
farm_code_sample = sample_farms_plot[0].replace('Farm/', '')
animal_ids_in_farm = [b for b in bottom_ids if b.startswith(farm_code_sample + '/')]
sample_animals_plot = np.random.choice(animal_ids_in_farm, min(5, len(animal_ids_in_farm)), replace=False)
for aid in sample_animals_plot:
    ts = Y_df[Y_df['unique_id'] == aid].sort_values('ds')
    axes[2].plot(ts['ds'], ts['y'], linewidth=1, alpha=0.8)
axes[2].set_title(f'Sample Animals from Farm {farm_code_sample}', fontsize=13, fontweight='bold')
axes[2].set_ylabel('kg')
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=30)

plt.tight_layout()
plt.savefig(OUT_DIR + 'hierarchy_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved ✅')

## 8. Filter: Keep Only Series with Enough History

StatsForecast needs a minimum number of observations per series.  
We drop bottom-level series with fewer than `MIN_OBS` months, then re-run `aggregate` on the filtered panel to stay coherent.

In [ ]:
MIN_OBS = 12   # minimum monthly observations required per bottom-level series

# Count observations per bottom series
obs_counts = (
    Y_df[Y_df['unique_id'].isin(bottom_ids)]
    .groupby('unique_id')['ds']
    .count()
)
valid_bottom = obs_counts[obs_counts >= MIN_OBS].index.tolist()
dropped = len(bottom_ids) - len(valid_bottom)
print(f'Bottom series with >= {MIN_OBS} months : {len(valid_bottom):,}  (dropped {dropped:,})')

# Filter the source monthly panel and re-run aggregate to keep coherence
if BOTTOM_LEVEL == 'farm':
    valid_farms = set(v for v in valid_bottom)
    df_agg_filtered = df_agg[df_agg['Farm_Code'].isin(valid_farms)].copy()
else:
    valid_animals = set(valid_bottom)
    df_agg_filtered = df_agg[df_agg['Animal'].isin(valid_animals)].copy()

Y_df, S_df, tags = aggregate(df_agg_filtered, spec)
Y_df = Y_df.reset_index(drop=True)

# Refresh convenience lists
all_ids    = Y_df['unique_id'].unique().tolist()
bottom_ids = tags[list(tags.keys())[-1]]
farm_ids   = tags.get('Farm_Code', [])

print(f'After filter → {Y_df["unique_id"].nunique():,} series, {len(Y_df):,} rows')
gc.collect()

In [ ]:
n_all    = len(all_ids)
n_bottom = len(bottom_ids)
id_to_col = {sid: j for j, sid in enumerate(bottom_ids)}
id_to_row = {sid: i for i, sid in enumerate(all_ids)}

farm_animals = defaultdict(list)
for bid in bottom_ids:
    farm_animals[bid.split('/')[0]].append(bid)

rows, cols = [], []
# Total
for j in range(n_bottom):
    rows.append(id_to_row['Total']); cols.append(j)
# Farms
for farm_code, animals in farm_animals.items():
    fuid = f'Farm/{farm_code}'
    if fuid not in id_to_row: continue
    frow = id_to_row[fuid]
    for aid in animals:
        rows.append(frow); cols.append(id_to_col[aid])
# Animals (identity)
for j, bid in enumerate(bottom_ids):
    rows.append(id_to_row[bid]); cols.append(j)

S_sparse = sp.csr_matrix(
    (np.ones(len(rows), dtype=np.float32), (rows, cols)),
    shape=(n_all, n_bottom)
)

S_df = pd.DataFrame(
    S_sparse.toarray(),
    index=all_ids,
    columns=bottom_ids,
    dtype=np.float32,
)
S_df.index.name = 'unique_id'
S_df = S_df.reset_index()

# Save
sp.save_npz(OUT_DIR + 'S_matrix.npz', S_sparse)
with open(OUT_DIR + 'tags.json', 'w') as f:
    json.dump(tags, f)
Y_df.to_parquet(OUT_DIR + 'Y_df.parquet', index=False)

print(f'S matrix shape: {S_sparse.shape}  ← (n_series × n_bottom_animals)')
print('S matrix, tags, Y_df saved ✅')

In [ ]:
cutoff = Y_df['ds'].max() - pd.DateOffset(months=H)

Y_train = Y_df[Y_df['ds'] <= cutoff].copy()
Y_test  = Y_df[Y_df['ds'] >  cutoff].copy()

print(f'Train: up to {cutoff.date()}  ({Y_train["ds"].nunique()} months)')
print(f'Test : {Y_test["ds"].min().date()} → {Y_test["ds"].max().date()}  ({Y_test["ds"].nunique()} months)')

## 11. Base Forecasts

We fit **AutoETS** (automatic exponential smoothing) independently on every series.  
This is the standard first step in any HTS pipeline — base forecasts are produced bottom-up, then reconciled.

We also include:
- `SeasonalNaive(season_length=12)` — a strong seasonal baseline  
- `AutoARIMA` — optional, slower but often more accurate

In [ ]:
models = [
    AutoETS(season_length=12),
    SeasonalNaive(season_length=12),
    # AutoARIMA(season_length=12),   # uncomment if you have time
]

sf = StatsForecast(
    models=models,
    freq=FREQ,
    n_jobs=-1,           # use all CPU cores
    fallback_model=SeasonalNaive(season_length=12),
)

print(f'Fitting {len(Y_train["unique_id"].nunique()):,} series...')
Y_hat_df = sf.forecast(df=Y_train, h=H, fitted=True)
Y_fitted_df = sf.forecast_fitted_values()

print('Base forecasts done ✅')
print(Y_hat_df.head())

In [ ]:
# Save base forecasts
Y_hat_df.to_parquet(OUT_DIR + 'Y_hat_base.parquet', index=False)
Y_fitted_df.to_parquet(OUT_DIR + 'Y_fitted_base.parquet', index=False)
print('Saved ✅')

## 12. Reconciliation

Raw base forecasts are **incoherent**: the Total forecast will not equal the sum of Farm forecasts.  
Reconciliation enforces coherence across all levels.

| Method | Description |
|---|---|
| **BottomUp** | Aggregate animal forecasts up. Ignores upper-level information. |
| **MinTrace(ols)** | OLS-based optimal reconciliation (Wickramasuriya 2019). |
| **MinTrace(mint_shrink)** | MinTrace with shrinkage covariance — often best in practice. |

In [ ]:
reconcilers = [
    BottomUp(),
    MinTrace(method='ols'),
    MinTrace(method='mint_shrink'),
]

hrec = HierarchicalReconciliation(reconcilers=reconcilers)

Y_rec_df = hrec.reconcile(
    Y_hat_df   = Y_hat_df,
    Y_df       = Y_train,
    S          = S_df,
    tags       = tags,
)

print('Reconciliation done ✅')
print(Y_rec_df.columns.tolist())
Y_rec_df.head()

In [ ]:
Y_rec_df.to_parquet(OUT_DIR + 'Y_reconciled.parquet', index=False)
print('Saved ✅')

## 13. Evaluation

We compute **RMSE** and **MAE** at each level of the hierarchy separately.  
This lets you see whether reconciliation improves accuracy at the farm and total level, at the cost of animal-level accuracy (or vice versa).

In [ ]:
from hierarchicalforecast.evaluation import HierarchicalEvaluation

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

evaluator = HierarchicalEvaluation(evaluators=[rmse, mae])

# Y_rec_df has columns: unique_id, ds, AutoETS, AutoETS/BottomUp, AutoETS/MinTrace_ols, ...
# Y_test needs: unique_id, ds, y
eval_results = evaluator.evaluate(
    Y_hat  = Y_rec_df,
    Y_test = Y_test,
    tags   = tags,
    benchmark = 'SeasonalNaive',   # compare all methods against this baseline
)

print('\n=== EVALUATION RESULTS ===')
print(eval_results.to_string())
eval_results.to_csv(OUT_DIR + 'evaluation_results.csv')
print('\nSaved ✅')

In [ ]:
# ── Thesis-ready summary table ─────────────────────────────────────────────────
# Pivot: rows = level, columns = (method, metric)
summary = eval_results.round(3)
print('\n=== THESIS SUMMARY TABLE ===')
print(summary)

## 14. Plot Reconciled vs Actual

In [ ]:
def plot_forecast(unique_id, Y_train, Y_test, Y_rec_df, model_col, ax, title=None):
    train = Y_train[Y_train['unique_id'] == unique_id].sort_values('ds')
    test  = Y_test[Y_test['unique_id'] == unique_id].sort_values('ds')
    fc    = Y_rec_df[Y_rec_df['unique_id'] == unique_id].sort_values('ds')

    # Show last 24 months of train + full test
    train_tail = train.tail(24)

    ax.plot(train_tail['ds'], train_tail['y'], color='steelblue', label='Train', linewidth=1.5)
    ax.plot(test['ds'], test['y'], color='black', label='Actual', linewidth=1.5, linestyle='--')
    if model_col in fc.columns:
        ax.plot(fc['ds'], fc[model_col], color='tomato', label=model_col, linewidth=1.5)
    ax.axvline(test['ds'].min(), color='grey', linestyle=':', linewidth=1)
    ax.set_title(title or unique_id, fontsize=10, fontweight='bold')
    ax.legend(fontsize=7)
    ax.set_ylabel('milk_kg')


# Identify the MinTrace_mint_shrink column name
reconciled_col = [c for c in Y_rec_df.columns if 'mint_shrink' in c and 'AutoETS' in c]
reconciled_col = reconciled_col[0] if reconciled_col else Y_rec_df.columns[-1]
print(f'Plotting column: {reconciled_col}')

# Plot: Total + 2 farms + 2 animals
np.random.seed(SEED)
plot_farm    = np.random.choice(farm_ids, 2, replace=False)
plot_animals = [np.random.choice([b for b in bottom_ids if b.startswith(f.replace('Farm/',''))], 1)[0]
                for f in plot_farm]

fig, axes = plt.subplots(5, 1, figsize=(14, 18), sharex=False)

plot_forecast('Total',         Y_train, Y_test, Y_rec_df, reconciled_col, axes[0], 'Total Herd')
plot_forecast(plot_farm[0],    Y_train, Y_test, Y_rec_df, reconciled_col, axes[1], f'Farm: {plot_farm[0]}')
plot_forecast(plot_farm[1],    Y_train, Y_test, Y_rec_df, reconciled_col, axes[2], f'Farm: {plot_farm[1]}')
plot_forecast(plot_animals[0], Y_train, Y_test, Y_rec_df, reconciled_col, axes[3], f'Animal: {plot_animals[0]}')
plot_forecast(plot_animals[1], Y_train, Y_test, Y_rec_df, reconciled_col, axes[4], f'Animal: {plot_animals[1]}')

plt.tight_layout()
plt.savefig(OUT_DIR + 'reconciled_forecasts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved ✅')

## 15. Cross-Validation (Optional but Recommended)

Rolling cross-validation gives a more robust estimate of forecast accuracy than a single train/test split.

In [ ]:
# Cross-validation on base models
# Each window forecasts H months ahead, stepping back N_WINDOWS times
Y_cv_df = sf.cross_validation(
    df       = Y_df,
    h        = H,
    n_windows = N_WINDOWS,
    step_size = H,
)

print('Cross-validation done ✅')
print(Y_cv_df.head())

In [ ]:
# RMSE per level per model (CV)
model_cols = [c for c in Y_cv_df.columns if c not in ['unique_id', 'ds', 'cutoff', 'y']]

results_cv = []
for level_name, level_ids in tags.items():
    sub = Y_cv_df[Y_cv_df['unique_id'].isin(level_ids)]
    for col in model_cols:
        r = np.sqrt(np.mean((sub['y'] - sub[col]) ** 2))
        m = np.mean(np.abs(sub['y'] - sub[col]))
        results_cv.append({'Level': level_name, 'Model': col, 'RMSE': round(r, 3), 'MAE': round(m, 3)})

cv_table = pd.DataFrame(results_cv).pivot(index='Level', columns='Model', values='RMSE')
print('\n=== CV RMSE per Level ===')
print(cv_table.to_string())
cv_table.to_csv(OUT_DIR + 'cv_rmse_per_level.csv')
print('Saved ✅')

## Summary

| Output file | Content |
|---|---|
| `Y_df.parquet` | Full hierarchical panel (all series, all dates) |
| `S_matrix.npz` | Sparse summing matrix |
| `tags.json` | Level → series_id mapping |
| `Y_hat_base.parquet` | Base (incoherent) forecasts |
| `Y_reconciled.parquet` | Reconciled forecasts (BottomUp, MinTrace-OLS, MinTrace-Shrink) |
| `evaluation_results.csv` | RMSE/MAE per level per method |
| `cv_rmse_per_level.csv` | Cross-validated RMSE per level |
| `hierarchy_overview.png` | Hierarchy visualisation |
| `reconciled_forecasts.png` | Forecast plots at all levels |